# 5. 5. Konsumenta B, który zapisuje wiadomości do bazy Neo4j jako dwa grafy:
a) (User)-[:USES]->(Device) # Użytkownik używa urządzenia

b) (User)-[:TRANSFER]->(User) # Użytkownik A wykonał transfer do użytkownika B
z właściwościami:
- amount
- timestamp
- title

In [1]:
import os, sys
import json
import pandas as pd
from kafka import KafkaConsumer
from neo4j import GraphDatabase



In [2]:


consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='reda_kafka_lab:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='neo4j-group',
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)


In [3]:
def run_cypher(driver, query: str):
    with driver.session() as session:
        result = session.run(query)
        records = list(result)
        
        if not records:
            print("Brak wyników.")
            return
        
        df = pd.DataFrame([r.data() for r in records])
        return df


def qexec(query: str):
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    
    try:
        df = run_cypher(driver, query)
    finally:
        driver.close()
    
    return df

In [4]:
# Połączenie z Neo4j
URI = "bolt://reda_neo4j_lab:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"  

In [5]:
#wyczysc wszystkie dane
query = """
MATCH (n)
DETACH DELETE n
"""

qexec(query)

Brak wyników.


In [6]:
#sprawdz czy baza jest czysta
query = """
MATCH (n)
RETURN count(n) AS nodes_count
"""

qexec(query)

,nodes_count
0,0


In [7]:
print(f"Waiting for transfer...")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
def save(tx, data):
    query = """
    MERGE(sender:User {id:$sender, role:'sender'})
    MERGE(receiver:User {id:$receiver, role:'receiver'})

    MERGE(device_sender:Device {id:$device_sender})
    MERGE(device_receiver:Device {id:$device_receiver})

    MERGE (sender)-[:USES {timestamp: datetime($timestamp)}]->(device_sender)
    MERGE (receiver)-[:USES {timestamp: datetime($timestamp)}]->(device_receiver)

    CREATE (sender)-[:TRANSFER {
        amount: $amount,
        timestamp: datetime($timestamp),
        title: $title
    }]->(receiver)
    """
    tx.run(query, **data)

try:
    with driver.session() as session:
        for msg in consumer:
            data = msg.value
            session.execute_write(save, data)
            print(f"Inserted into Neo4j: {data}")

finally:
    driver.close()

print(f"Inserted into DB: {data}")

Waiting for transfer...
Inserted into Neo4j: {'sender': 'user1', 'receiver': 'user2', 'amount': 120.5, 'timestamp': '2026-05-10T10:00:00', 'device_sender': 'devA', 'device_receiver': 'devB', 'title': 'zwrot za obiad'}
Inserted into Neo4j: {'sender': 'user3', 'receiver': 'user4', 'amount': 75.0, 'timestamp': '2026-05-09T11:00:00', 'device_sender': 'devC', 'device_receiver': 'devD', 'title': 'oddaje pieniadze'}
Inserted into Neo4j: {'sender': 'user5', 'receiver': 'user6', 'amount': 200.0, 'timestamp': '2026-05-08T09:30:00', 'device_sender': 'devE', 'device_receiver': 'devF', 'title': 'zwrot kosztow'}
Inserted into Neo4j: {'sender': 'user2', 'receiver': 'user1', 'amount': 50.0, 'timestamp': '2026-05-07T14:00:00', 'device_sender': 'devB', 'device_receiver': 'devA', 'title': 'zwrot za paliwo'}
Inserted into Neo4j: {'sender': 'user7', 'receiver': 'user8', 'amount': 300.0, 'timestamp': '2026-05-06T16:00:00', 'device_sender': 'devG', 'device_receiver': 'devH', 'title': 'oddaje dlug'}
Inserted 

KeyboardInterrupt: 

# 6. Zapytania sprawdzające:
b) (Cypher) Znajdź użytkowników korzystających z tego samego urządzenia w ciągu
ostatnich 20 dni.

In [8]:
#b) (Cypher) Znajdź użytkowników korzystających z tego samego urządzenia w ciągu
#ostatnich 3 dni. 
# UWAGA - zmiana na 20 dni bo 3 dni wyjechalo poza okienko :) i byl brak wynikow


query = """
MATCH (u1:User)-[uses1:USES]->(d:Device)<-[uses2:USES]-(u2:User)
WHERE u1.id < u2.id
  AND uses1.timestamp >= datetime() - duration({days: 20})
  AND uses2.timestamp >= datetime() - duration({days: 20})
RETURN DISTINCT u1.id as user1, u2.id as user2, d.id as device;
"""

qexec(query)

,user1,user2,device
0,user1,user12,devA
1,user12,user16,devA
2,user1,user16,devA
3,user13,user18,devB
4,user13,user2,devB
5,user18,user2,devB
6,user14,user3,devC
7,user15,user4,devD
8,user16,user5,devE
9,user17,user6,devF
